# Prédiction sur 24 heures de l'aqi

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import streamlit as st
from supabase import create_client
from sklearn.metrics import r2_score

## Connexion à la Base de donnée via l'api

In [2]:
url = "https://wynjcciiprpxtkvjaurm.supabase.co"
key = key = st.secrets["supabase_tok"]
supabase = create_client(url, key)

## Récupération de la base table "mesures"

In [3]:
response = supabase.table("mesures").select("*").execute()
lignes = response.data

print(lignes)

[{'id': 715614, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'longitude': 1.8664, 'time': '2026-07-31T20:00:00', 'european_aqi': 33, 'pm10': 5.8, 'pm2_5': 3.9, 'nitrogen_dioxide': 1.7, 'ozone': 85, 'sulphur_dioxide': 0.6, 'carbon_monoxide': 124, 'birch_pollen': 0, 'temperature_2m': 24.6, 'relative_humidity_2m': 47, 'wind_speed_10m': 12.7, 'wind_direction_10m': 322, 'precipitation': 0}, {'id': 715615, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'longitude': 1.8664, 'time': '2026-07-31T21:00:00', 'european_aqi': 33, 'pm10': 5.9, 'pm2_5': 4.1, 'nitrogen_dioxide': 2, 'ozone': 85, 'sulphur_dioxide': 0.5, 'carbon_monoxide': 129, 'birch_pollen': 0, 'temperature_2m': 23.2, 'relative_humidity_2m': 50, 'wind_speed_10m': 14.1, 'wind_direction_10m': 332, 'precipitation': 0}, {'id': 715616, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'lo

### Test

In [4]:
response = supabase.table("mesures").select("*").execute()

# Récupérer toutes les lignes retournées
lignes = response.data

print(lignes)

[{'id': 715614, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'longitude': 1.8664, 'time': '2026-07-31T20:00:00', 'european_aqi': 33, 'pm10': 5.8, 'pm2_5': 3.9, 'nitrogen_dioxide': 1.7, 'ozone': 85, 'sulphur_dioxide': 0.6, 'carbon_monoxide': 124, 'birch_pollen': 0, 'temperature_2m': 24.6, 'relative_humidity_2m': 47, 'wind_speed_10m': 12.7, 'wind_direction_10m': 322, 'precipitation': 0}, {'id': 715615, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'longitude': 1.8664, 'time': '2026-07-31T21:00:00', 'european_aqi': 33, 'pm10': 5.9, 'pm2_5': 4.1, 'nitrogen_dioxide': 2, 'ozone': 85, 'sulphur_dioxide': 0.5, 'carbon_monoxide': 129, 'birch_pollen': 0, 'temperature_2m': 23.2, 'relative_humidity_2m': 50, 'wind_speed_10m': 14.1, 'wind_direction_10m': 332, 'precipitation': 0}, {'id': 715616, 'capteur_id': '3106', 'capteur_nom': 'Zone Rurale Nord-ouest - Fremainville', 'latitude': 49.0631, 'lo

### code

In [5]:
supabase = create_client(url, key)

toutes_les_lignes = []
debut = 0
taille_page = 1000

while True:
    response = (
        supabase.table("mesures")
        .select("*")
        .range(debut, debut + taille_page - 1)
        .execute()
    )

    lignes = response.data
    toutes_les_lignes.extend(lignes)

    if len(lignes) < taille_page:
        break

    debut += taille_page

df = pd.DataFrame(toutes_les_lignes)

print("Nombre total de lignes :", len(df))



Nombre total de lignes : 63301


In [6]:
df

,id,capteur_id,capteur_nom,latitude,longitude,time,european_aqi,pm10,pm2_5,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide,birch_pollen,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation
0,715614,3106,Zone Rurale Nord-ouest - Fremainville,49.063100,1.866400,2026-07-31T20:00:00,33,5.8,3.9,1.7,85,0.6,124,0,24.6,47,12.7,322,0.0
1,715615,3106,Zone Rurale Nord-ouest - Fremainville,49.063100,1.866400,2026-07-31T21:00:00,33,5.9,4.1,2.0,85,0.5,129,0,23.2,50,14.1,332,0.0
2,715616,3106,Zone Rurale Nord-ouest - Fremainville,49.063100,1.866400,2026-07-31T22:00:00,32,9.1,4.7,2.0,83,0.3,129,0,21.0,65,15.1,345,0.0
3,715617,3106,Zone Rurale Nord-ouest - Fremainville,49.063100,1.866400,2026-07-31T23:00:00,30,9.0,5.1,2.1,79,0.3,129,0,19.7,72,13.5,337,0.0
4,715618,3106,Zone Rurale Nord-ouest - Fremainville,49.063100,1.866400,2026-08-01T00:00:00,28,8.9,5.3,2.2,76,0.3,131,0,18.6,79,10.7,335,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63296,780728,9-1454125,Rue Adrien Damalix,48.820923,2.421895,2026-10-17T10:00:00,25,8.7,6.1,13.6,46,1.6,212,0,10.6,79,11.8,340,0.0
63297,780729,103094,Rd934 Coulommiers,48.805600,3.084440,2026-10-17T10:00:00,18,9.1,4.4,3.0,55,0.4,165,0,10.2,85,15.6,352,0.0
63298,780734,153093,Lognes,48.840300,2.634630,2026-10-17T10:00:00,21,8.8,5.4,7.9,50,1.0,195,0,9.9,86,11.4,347,0.0
63299,780739,203098,Autoroute A1 Saint-denis,48.925100,2.356540,2026-10-17T10:00:00,22,10.5,5.8,8.8,48,1.3,202,0,10.8,79,8.5,348,0.0


### Statistique sur les données recceuillis

In [7]:
df.describe()
df2= df
df["time"].dtype

<StringDtype(na_value=nan)>

### Préparations, ajout de colonnes, features ingeneering

In [8]:


# --- 0. Préparation ---
df = df2.copy()
######################conversion et réorganisation par id##################""
df["time"] = (pd.to_datetime(df["time"])
                .dt.tz_localize("Europe/Paris", ambiguous="NaT", nonexistent="shift_forward")# gère le passage à l'heure d'hiver
                .dt.tz_convert("UTC")
)


df = df.dropna(subset=["time"])                        # heures ambiguës (NaT)
df = df.drop_duplicates(["capteur_id", "time"])        # sécurité pour supprimer un capteur qui affiche des résultats à la meme date

# grille horaire régulière par capteur (heures manquantes -> NaN explicites)
df = (df.set_index("time")
        .groupby("capteur_id", observed=True)
        .resample("1h").asfreq()
        .drop(columns="capteur_id", errors="ignore")
        .reset_index())
df = df.sort_values(["capteur_id", "time"]).reset_index(drop=True)

# contrôle : doit afficher uniquement 0 days 01:00:00
print(df.groupby("capteur_id", observed=True)["time"].diff().value_counts().head())

# --- 1. Vent en composantes ---
rad = np.deg2rad(df["wind_direction_10m"])
df["wind_u"] = -df["wind_speed_10m"] * np.sin(rad)   # composante est-ouest
df["wind_v"] = -df["wind_speed_10m"] * np.cos(rad)   # composante nord-sud

g = df.groupby("capteur_id", observed=True)          # défini une seule fois, après les nouvelles colonnes

# --- 2. Passé des polluants ---
polluants = ["european_aqi", "pm10", "pm2_5", "nitrogen_dioxide",
             "ozone", "sulphur_dioxide", "carbon_monoxide", "birch_pollen"]
temps_pass = [1, 2, 3, 6, 12, 24, 48, 72, 168]   #pour le passé en heure
intervalle = [6, 24, 72] #pour l'interlle de temps en heure
################calcul des statistiques glissantes###################
f = {}
for pol in polluants:
    f[f"{pol}_pass0"] = df[pol]
    for t in temps_pass:
        f[f"{pol}_pass{t}"] = g[pol].shift(t)

    for i in intervalle:
        roll = g[pol].rolling(i, min_periods=i // 2)
        f[f"{pol}_mean{i}"] = roll.mean().reset_index(level=0, drop=True)
        f[f"{pol}_std{i}"]  = roll.std().reset_index(level=0, drop=True)
        f[f"{pol}_max{i}"]  = roll.max().reset_index(level=0, drop=True)

    f[f"{pol}_dif6"]  = df[pol] - g[pol].shift(6)
    f[f"{pol}_dif24"] = df[pol] - g[pol].shift(24)
historique = pd.DataFrame(f).astype("float32") #transforme f en dataframe

##################################### --- 3. Passé météo ---#####################################""
meteo = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m",
         "wind_u", "wind_v", "precipitation"]
met = {}
for m in meteo:
    met[f"{m}_pass0"] = df[m]
    for t in [3, 6, 12, 24]:
        met[f"{m}_pass{t}"] = g[m].shift(t)
past_meteo = pd.DataFrame(met).astype("float32")

# ---  Météo future, dépendant de h ---
def features_futur(h): # ajouter pour une ligne les éléments de la hème heure
    out = {}
    for v in meteo:
        fen = pd.concat([g[v].shift(-k) for k in range(1, h + 1)], axis=1)
        out[f"{v}_fut_h"]    = g[v].shift(-h) #la features méteo apres h heure
        out[f"{v}_fut_mean"] = fen.mean(axis=1) #la moyenne des futures meteo apres h heures
        out[f"{v}_fut_min"]  = fen.min(axis=1)# le minimum des futures après h heures
        out[f"{v}_fut_max"]  = fen.max(axis=1) # le max
        out[f"{v}_delta"]    = out[f"{v}_fut_h"] - df[v]# la différence
    pr = pd.concat([g["precipitation"].shift(-k) for k in range(1, h + 1)], axis=1)
    out["precip_cumul"] = pr.sum(axis=1, min_count=h)
    d = (df["time"] + pd.Timedelta(hours=h)).dt.tz_convert("Europe/Paris")
    out["hour_sin"] = np.sin(2 * np.pi * d.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * d.dt.hour / 24)
    out["dow"], out["month"] = d.dt.dayofweek, d.dt.month
    return pd.DataFrame(out).astype("float32")



time
0 days 01:00:00    63306
Name: count, dtype: int64


In [9]:
# Dimensions et types
print(df.shape)
print(df.dtypes)

# Nombre de capteurs
print(len(df["capteur_id"].unique()))

# Doublons capteur-date
print(
    df.duplicated(["capteur_id", "time"]).sum()
)

# Valeurs manquantes
print(
    df[polluants + meteo].isna().sum().sort_values(
        ascending=False
    )
)

# Intervalles entre mesures chaque  heures
ecarts = (
    df.groupby("capteur_id")["time"]
      .diff()   
)
print(ecarts.head(500))



(63340, 21)
capteur_id                              str
time                    datetime64[us, UTC]
id                                  float64
capteur_nom                             str
latitude                            float64
longitude                           float64
european_aqi                        float64
pm10                                float64
pm2_5                               float64
nitrogen_dioxide                    float64
ozone                               float64
sulphur_dioxide                     float64
carbon_monoxide                     float64
birch_pollen                        float64
temperature_2m                      float64
relative_humidity_2m                float64
wind_speed_10m                      float64
wind_direction_10m                  float64
precipitation                       float64
wind_u                              float64
wind_v                              float64
dtype: object
34
0
european_aqi            39
pm10              

# debut du machine learning

In [10]:
###Assemblage du dataset
h=24
y = g["european_aqi"].shift(-h) #target cible
X= pd.concat([historique,past_meteo,features_futur(h)],axis = 1) #concaténation de nos différentes colonnes et nouvelles variables
X["capteur_id"] = df["capteur_id"].astype("category") 
X["time"] = df["time"] #ajouter le temps

In [11]:
mask = y.notna()# mask boolean qui renvoie true à chaque fois qu'une valeur est présente dans la colonne et false sinon
X, y = X[mask], y[mask] # filtrage par le masque

In [12]:
# filtrage par le temps ici on filtre par le temps et non en désordre
cut_test  = X["time"].quantile(0.85) # valeur hors de 85% de valeur générale
cut_val   = X["time"].quantile(0.70) # valeur hors de 75% de valeur générale
train = X["time"] < cut_val #ensemble d'entraiement
val   = (X["time"] >= cut_val) & (X["time"] < cut_test)# validation
test  = X["time"] >= cut_test #ensemble de test


In [13]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                          num_leaves=63, subsample=0.8, colsample_bytree=0.7) # model
model.fit(X[train].drop(columns="time"), y[train],
          eval_set=[(X[val].drop(columns="time"), y[val])],
          callbacks=[lgb.early_stopping(100)]) # arret au meilleur moment

c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.184219 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40215
[LightGBM] [Info] Number of data points in the train set: 43724, number of used features: 232
[LightGBM] [Info] Start training from score 28.774220
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[135]	valid_0's l2: 54.2526


,num_leaves,63
,learning_rate,0.03
,n_estimators,2000
,subsample,0.8
,colsample_bytree,0.7
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [14]:
X_val = X[val].drop(columns="time")

y_pred = model.predict(X_val)

mae = mean_absolute_error(y[val], y_pred)
mse = mean_squared_error(y[val], y_pred)
rmse = mse ** 0.5

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")

MAE  : 5.7039
MSE  : 54.2526
RMSE : 7.3656


In [15]:
import lightgbm as lgb

params = dict(n_estimators=3000, learning_rate=0.02, num_leaves=15,
              min_child_samples=200, subsample=0.7, subsample_freq=1,
              colsample_bytree=0.3, reg_lambda=20, n_jobs=-1, verbose=-1)

t = df["time"]
models = {}
feats = {}    # features de chaque horizon, pour pouvoir prédire ensuite

for h in range(1, 25):
    y = g["european_aqi"].shift(-h)
    X = pd.concat([historique, past_meteo, features_futur(h)], axis=1)

    mask = y.notna()
    Xm, ym, tm = X[mask], y[mask], t[mask]

    cut_val, cut_test = tm.quantile(0.70), tm.quantile(0.85)
    gap = pd.Timedelta(hours=h)
    train = tm < cut_val - gap
    val   = (tm >= cut_val) & (tm < cut_test - gap)

    m = lgb.LGBMRegressor(**params)
    m.fit(Xm[train], ym[train], eval_set=[(Xm[val], ym[val])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    models[h] = m
    feats[h] = X            # on garde X complet (avec les lignes sans cible)
    print(h, "ok, itérations :", m.best_iteration_)

c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


1 ok, itérations : 1869


c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


2 ok, itérations : 1314


c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

22 ok, itérations : 229


c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


23 ok, itérations : 238


c:\Users\BEBETO\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


24 ok, itérations : 335


In [16]:
cap = "103094"
idx = df.index[df["capteur_id"] == cap]
i = idx[-25]                    # instant "maintenant" simulé (les 24 h suivantes sont connues)

res = []
for h in range(1, 25):
    res.append({
        "heure_locale": (df.loc[i, "time"] + pd.Timedelta(hours=h)).tz_convert("Europe/Paris"),
        "aqi_predit": float(models[h].predict(feats[h].loc[[i]])[0]),
        "aqi_reel": df.loc[i + h, "european_aqi"],   # lignes contiguës grâce à la grille horaire
    })

prev = pd.DataFrame(res)
print(prev.round(1).to_string(index=False))
print("MAE :", (prev.aqi_predit - prev.aqi_reel).abs().mean())
print("RMSE:",mean_squared_error(prev.aqi_predit , prev.aqi_reel))

             heure_locale  aqi_predit  aqi_reel
2026-10-16 11:00:00+02:00        24.6      29.0
2026-10-16 12:00:00+02:00        23.5      26.0
2026-10-16 13:00:00+02:00        24.8      23.0
2026-10-16 14:00:00+02:00        25.2      23.0
2026-10-16 15:00:00+02:00        25.4      22.0
2026-10-16 16:00:00+02:00        25.7      26.0
2026-10-16 17:00:00+02:00        25.5      25.0
2026-10-16 18:00:00+02:00        24.1      23.0
2026-10-16 19:00:00+02:00        22.5      19.0
2026-10-16 20:00:00+02:00        22.0      18.0
2026-10-16 21:00:00+02:00        21.6      17.0
2026-10-16 22:00:00+02:00        22.1      19.0
2026-10-16 23:00:00+02:00        21.8      19.0
2026-10-17 00:00:00+02:00        20.9      19.0
2026-10-17 01:00:00+02:00        20.7      20.0
2026-10-17 02:00:00+02:00        19.5      19.0
2026-10-17 03:00:00+02:00        20.6      21.0
2026-10-17 04:00:00+02:00        21.0      19.0
2026-10-17 05:00:00+02:00        20.4      19.0
2026-10-17 06:00:00+02:00        20.9   

C:\Users\BEBETO\AppData\Local\Temp\ipykernel_18996\1640823.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(prev.round(1).to_string(index=False))


# test de prédiction pour 1h avec un capteur

In [17]:
from datetime import datetime,timedelta
from zoneinfo import ZoneInfo


date_heure = datetime.now(ZoneInfo("Europe/Paris")).replace(
    minute=0, second=0, microsecond=0
)

cap = "115722"
idx = df.index[df["capteur_id"] == cap]
i = idx[-25]                    # instant "maintenant" simulé (les 24 h suivantes sont connues)

res = []
for h in range(1, 25):
    res.append({
        "heure_locale": (date_heure+timedelta(hours=h)),
        "aqi_predit": float(models[h].predict(feats[h].loc[[i]])[0]),
           # lignes contiguës grâce à la grille horaire
    })

prev = pd.DataFrame(res)
print(prev.round(1).to_string(index=False))
print("moyenne :", (prev.aqi_predit ).mean())

             heure_locale  aqi_predit
2026-10-17 12:00:00+02:00        40.4
2026-10-17 13:00:00+02:00        28.1
2026-10-17 14:00:00+02:00        25.9
2026-10-17 15:00:00+02:00        27.1
2026-10-17 16:00:00+02:00        27.2
2026-10-17 17:00:00+02:00        27.6
2026-10-17 18:00:00+02:00        28.4
2026-10-17 19:00:00+02:00        26.3
2026-10-17 20:00:00+02:00        26.0
2026-10-17 21:00:00+02:00        27.7
2026-10-17 22:00:00+02:00        32.2
2026-10-17 23:00:00+02:00        32.9
2026-10-18 00:00:00+02:00        34.0
2026-10-18 01:00:00+02:00        30.9
2026-10-18 02:00:00+02:00        28.5
2026-10-18 03:00:00+02:00        26.7
2026-10-18 04:00:00+02:00        26.4
2026-10-18 05:00:00+02:00        25.4
2026-10-18 06:00:00+02:00        27.4
2026-10-18 07:00:00+02:00        28.0
2026-10-18 08:00:00+02:00        31.7
2026-10-18 09:00:00+02:00        36.3
2026-10-18 10:00:00+02:00        39.1
2026-10-18 11:00:00+02:00        41.0
moyenne : 30.229829963324807


C:\Users\BEBETO\AppData\Local\Temp\ipykernel_18996\913778909.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(prev.round(1).to_string(index=False))


In [18]:

from datetime import datetime,timedelta
from zoneinfo import ZoneInfo
from dico import capteurs
previs={}

date_heure = datetime.now(ZoneInfo("Europe/Paris")).replace(
    minute=0, second=0, microsecond=0
)
for capteur in capteurs:
    cap = capteur["id"]
    idx = df.index[df["capteur_id"] == cap]
    i = idx[-25]                    # instant "maintenant" simulé (les 24 h suivantes sont connues)
    
    res = []
    for h in range(1, 25):
        res.append({
            "heure_locale": (date_heure+timedelta(hours=h)),
            "aqi_predit": float(models[h].predict(feats[h].loc[[i]])[0]),
            # lignes contiguës grâce à la grille horaire
        })
    previs[cap]=res

previs = pd.DataFrame(previs)
print(previs)


                                                 3106  \
0   {'heure_locale': 2026-10-17 12:00:00+02:00, 'a...   
1   {'heure_locale': 2026-10-17 13:00:00+02:00, 'a...   
2   {'heure_locale': 2026-10-17 14:00:00+02:00, 'a...   
3   {'heure_locale': 2026-10-17 15:00:00+02:00, 'a...   
4   {'heure_locale': 2026-10-17 16:00:00+02:00, 'a...   
5   {'heure_locale': 2026-10-17 17:00:00+02:00, 'a...   
6   {'heure_locale': 2026-10-17 18:00:00+02:00, 'a...   
7   {'heure_locale': 2026-10-17 19:00:00+02:00, 'a...   
8   {'heure_locale': 2026-10-17 20:00:00+02:00, 'a...   
9   {'heure_locale': 2026-10-17 21:00:00+02:00, 'a...   
10  {'heure_locale': 2026-10-17 22:00:00+02:00, 'a...   
11  {'heure_locale': 2026-10-17 23:00:00+02:00, 'a...   
12  {'heure_locale': 2026-10-18 00:00:00+02:00, 'a...   
13  {'heure_locale': 2026-10-18 01:00:00+02:00, 'a...   
14  {'heure_locale': 2026-10-18 02:00:00+02:00, 'a...   
15  {'heure_locale': 2026-10-18 03:00:00+02:00, 'a...   
16  {'heure_locale': 2026-10-18

In [19]:
import joblib, json
from pathlib import Path

Path("modele_aqi").mkdir(exist_ok=True)

bundle = {
    "models": models,                                   # {h: LGBMRegressor}
    "columns": {h: list(feats[h].columns) for h in models},
    "best_iteration": {h: models[h].best_iteration_ for h in models},
    "config": {
        "target": "european_aqi",
        "horizons": list(range(1, 25)),
        "polluants": polluants,
        "temps_pass": temps_pass,
        "intervalle": intervalle,
        "meteo": meteo,
        "train_end": str(t.max()),
        "lightgbm_version": lgb.__version__,
    },
}
joblib.dump(bundle, "modele_aqi/bundle_aqi.joblib", compress=3)

['modele_aqi/bundle_aqi.joblib']

In [20]:
bundle = joblib.load("modele_aqi/bundle_aqi.joblib")
models = bundle["models"]
columns = bundle["columns"]

# prédiction : toujours réordonner les colonnes comme à l'entraînement
#pred = models[12].predict(X_nouveau[columns[12]])